# 時系列分析: 甘口・フルーティ傾向は強まっているか？

SAKETIMEランキング上位銘柄の「甘口・フルーティ」傾向が年々強まっているかを、
構造化データ（テイスト選択）とレビューテキストの両面から検証する。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Hiragino Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

ranking = pd.read_csv('../data/ranking.csv')
reviews = pd.read_csv('../data/reviews.csv')
rev = reviews.merge(ranking[['brand_id','rank','name']], on='brand_id', how='left')

def parse_date(d):
    m = re.match(r'(\d{4})年(\d{1,2})月(\d{1,2})日', str(d))
    if m:
        return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))
    return pd.NaT

rev['date_parsed'] = rev['date'].apply(parse_date)
rev = rev.dropna(subset=['date_parsed']).copy()
rev['year'] = rev['date_parsed'].dt.year
rev = rev[(rev['year'] >= 2017) & (rev['year'] <= 2025)]

years = list(range(2017, 2026))
print(f'分析対象: {len(rev):,}件 (2017-2025)')

## 1. 構造化データの時系列推移

In [ ]:
def calc_taste_rates(data, years):
    results = {'sweet': [], 'dry': [], 'rich': [], 'light': []}
    for year in years:
        y = data[data['year'] == year]
        s = y.dropna(subset=['sweetness'])
        b = y.dropna(subset=['body'])
        results['sweet'].append(s['sweetness'].isin(['甘い+1','甘い+2']).mean() * 100 if len(s) > 50 else np.nan)
        results['dry'].append(s['sweetness'].isin(['辛い+1','辛い+2']).mean() * 100 if len(s) > 50 else np.nan)
        results['rich'].append(b['body'].isin(['重い+1','重い+2']).mean() * 100 if len(b) > 50 else np.nan)
        results['light'].append(b['body'].isin(['軽い+1','軽い+2']).mean() * 100 if len(b) > 50 else np.nan)
    return results

all_rates = calc_taste_rates(rev, years)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 甘口率の推移（全銘柄のみ）
axes[0,0].plot(years, all_rates['sweet'], 'o-', label='全銘柄', color='#E91E63', linewidth=2.5)
axes[0,0].set_title('甘口率の推移（甘い+1, +2）', fontsize=12)
axes[0,0].set_ylabel('%')
axes[0,0].legend()
axes[0,0].set_ylim(30, 60)

# 辛口率の推移（全銘柄のみ、Y軸スケール修正）
axes[0,1].plot(years, all_rates['dry'], 'o-', label='全銘柄', color='#2196F3', linewidth=2.5)
axes[0,1].set_title('辛口率の推移（辛い+1, +2）', fontsize=12)
axes[0,1].set_ylabel('%')
axes[0,1].legend()
axes[0,1].set_ylim(0, 25)

# 濃醇率の推移（全銘柄のみ）
axes[1,0].plot(years, all_rates['rich'], 'o-', label='全銘柄', color='#FF9800', linewidth=2.5)
axes[1,0].set_title('濃醇率の推移（重い+1, +2）', fontsize=12)
axes[1,0].set_ylabel('%')
axes[1,0].legend()
axes[1,0].set_ylim(0, 25)

# 軽快率の推移（全銘柄のみ）
axes[1,1].plot(years, all_rates['light'], 'o-', label='全銘柄', color='#4CAF50', linewidth=2.5)
axes[1,1].set_title('軽快率の推移（軽い+1, +2）', fontsize=12)
axes[1,1].set_ylabel('%')
axes[1,1].legend()
axes[1,1].set_ylim(20, 40)

for ax in axes.flat:
    ax.set_xlabel('年')
    ax.set_xticks(years)

plt.suptitle('構造化データによるテイスト傾向の時系列推移（全銘柄）', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../data/trend_structured.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. テキスト分析の時系列推移

In [ ]:
rev_c = rev.dropna(subset=['comment'])

kw_groups = {
    '甘口系': '甘い|甘み|甘味|甘口|甘さ|甘やか',
    '辛口系': '辛い|辛口|ドライ',
    '芳醇系': '芳醇|旨味|旨み|濃厚|コク|リッチ',
    '軽快系': 'すっきり|スッキリ|爽やか|クリア|軽い|軽快',
    'フルーティ系': 'フルーティ|フルーツ|果実|メロン|りんご|パイナップル|マスカット|マンゴー|桃|バナナ|ライチ|洋梨',
}

colors = {'甘口系': '#E91E63', '辛口系': '#2196F3', '芳醇系': '#FF9800', 
          '軽快系': '#4CAF50', 'フルーティ系': '#9C27B0'}

fig, ax = plt.subplots(figsize=(14, 7))

for label, pattern in kw_groups.items():
    rates_all = []
    for year in years:
        y_all = rev_c[rev_c['year'] == year]
        rates_all.append(y_all['comment'].str.contains(pattern, na=False).mean() * 100)
    
    ax.plot(years, rates_all, 'o-', label=label, color=colors[label], linewidth=2.5)

ax.set_title('全銘柄のキーワード出現率推移', fontsize=14)
ax.set_xlabel('年')
ax.set_ylabel('出現率 (%)')
ax.legend(fontsize=11)
ax.set_xticks(years)
ax.set_ylim(0, 40)

plt.suptitle('レビューテキスト中の味わいキーワード出現率の推移（全銘柄）', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../data/trend_text.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 甘口とフルーティの乖離: 上位50位 vs 全体の差の推移

In [ ]:
# 上位50位と全体の「差」がどう変化しているか（上位の偏りが強まっているか）
fig, ax = plt.subplots(figsize=(12, 6))

for label, pattern in kw_groups.items():
    diffs = []
    for year in years:
        y_all = rev_c[rev_c['year'] == year]
        y_top = rev_c[(rev_c['year'] == year) & (rev_c['rank'] <= 50)]
        if len(y_top) > 100:
            r_all = y_all['comment'].str.contains(pattern, na=False).mean() * 100
            r_top = y_top['comment'].str.contains(pattern, na=False).mean() * 100
            diffs.append(r_top - r_all)
        else:
            diffs.append(np.nan)
    ax.plot(years, diffs, 'o-', label=label, color=colors[label], linewidth=2)

ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('年', fontsize=11)
ax.set_ylabel('差 (上位50位 - 全体) [ポイント]', fontsize=11)
ax.set_title('上位50位と全体のキーワード出現率の差の推移\n（正の値 = 上位ほどその傾向が強い）', fontsize=13)
ax.legend()
ax.set_xticks(years)

plt.tight_layout()
plt.savefig('../data/trend_gap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 構造化データでも同様に差の推移を確認

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

taste_labels = {'甘口率': ('sweetness', ['甘い+1','甘い+2'], '#E91E63'),
                '辛口率': ('sweetness', ['辛い+1','辛い+2'], '#2196F3'),
                '濃醇率': ('body', ['重い+1','重い+2'], '#FF9800'),
                '軽快率': ('body', ['軽い+1','軽い+2'], '#4CAF50')}

for label, (col, vals, color) in taste_labels.items():
    diffs = []
    for year in years:
        y_all = rev[rev['year'] == year].dropna(subset=[col])
        y_top = rev[(rev['year'] == year) & (rev['rank'] <= 50)].dropna(subset=[col])
        if len(y_top) > 50:
            r_all = y_all[col].isin(vals).mean() * 100
            r_top = y_top[col].isin(vals).mean() * 100
            diffs.append(r_top - r_all)
        else:
            diffs.append(np.nan)
    ax.plot(years, diffs, 'o-', label=label, color=color, linewidth=2)

ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('年', fontsize=11)
ax.set_ylabel('差 (上位50位 - 全体) [ポイント]', fontsize=11)
ax.set_title('構造化データ: 上位50位と全体のテイスト率の差の推移\n（正 = 上位にその傾向が偏っている）', fontsize=13)
ax.legend()
ax.set_xticks(years)

plt.tight_layout()
plt.savefig('../data/trend_gap_structured.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 結論

In [ ]:
print('=' * 70)
print('時系列分析の結論')
print('=' * 70)
print('''
■ 全体的なトレンド（2017→2025）
  - 辛口率は全体で下降傾向（15% → 11%）、上位50位でも減少（10% → 6%）
  - 濃醇率は全体・上位ともに下降（17% → 10%, 上位: 17% → 8%）
  - 軽快率はやや上昇傾向（上位50位: 30% → 32-37%）
  - フルーティ系キーワードは2017→2021で急増、その後やや横ばい〜再上昇

■ 甘口傾向は強まっているか？
  - 構造化データの甘口率は意外にも安定（上位50位: 54-60%で推移）
  - テキストの甘口系キーワードは2017→2019で大幅増加（30%→36%）、
    その後は30-36%で安定
  → 2019年頃に一段階上がり、以降は横ばい。現在進行形で強まっているとは言えない

■ フルーティ傾向は強まっているか？
  - テキストでは2017年の18% → 2025年の28%（上位50位）へ明確に上昇
  - 特に2020-2021年にピーク、2022年に一時低下後、2024-2025で再上昇
  → フルーティ傾向は明確に強まっている

■ まとめ
  日本酒の味わいトレンドとして最も顕著なのは:
  1. 辛口・濃醇の減少（全体的に「重い酒」離れ）
  2. フルーティ表現の増加（果実系の香味表現が定着）
  3. 甘口は高水準で安定（急激な変化ではなく既に主流化）
  4. 上位銘柄の「軽快＋フルーティ」志向が年々強まっている
''')